# moe-sae discovery run (spec §20 step 4) — Colab Pro A100-40

All state lives in GCS; **this notebook is rerunnable** — if the session dies at any point, just run all cells again in a fresh session and the chain resumes where it stopped (capture checkpoints every 100 examples; training runs skip-if-complete).

Setup once: add a Colab secret named `GITHUB_TOKEN` (repo-read PAT) via the 🔑 panel, and set `BUCKET` below to a bucket you own (same project you'll bill GCS storage to, ~$15/mo for 770 GB).

In [ ]:
gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(gpu)
assert any("A100" in g for g in gpu), "Runtime > Change runtime type > A100 GPU"

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()  # GCS access for the state bucket
os.environ["BUCKET"] = "gs://CHANGE-ME/discovery"  # <-- set your bucket

if not os.path.exists("moe-sae"):
    token = userdata.get("GITHUB_TOKEN")
    !git clone -q https://{token}@github.com/andrewbistras/moe-sae.git
%cd moe-sae
!git pull -q

In [ ]:
# Env: keep Colab's CUDA torch; pin transformers for the vendored model code.
!pip -q install -e . --no-deps
!pip -q install "transformers==4.38.2" timm einops sentencepiece safetensors datasets accelerate pyyaml
!python -c "import torch, moe_sae; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())"

In [ ]:
# The whole chain: capture (3 workers) → merge → stats → 14 runs → eval → gate.
# Rerunnable; watch logs/capture_w*.log for worker progress.
!bash scripts/cloud/colab_discovery.sh

In [ ]:
import json, pathlib
gate = pathlib.Path("reports/sanity_gate_discovery.json")
if gate.exists():
    print(json.dumps(json.loads(gate.read_text()), indent=2))
else:
    print("gate not reached yet — rerun the previous cell (it resumes)")